# 🗂️ Subset & Model Selection
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When you have many predictors, which ones should you include in the model? Subset selection methods systematically compare model subsets. Information criteria (Cp, BIC, adjusted R²) give principled ways to choose model size without a test set.

### What you'll learn
- Best subset selection: exhaustive search
- Forward and backward stepwise selection: greedy approximations
- Cp, BIC, and adjusted R² as model selection criteria
- Practical comparison with cross-validation

### Dataset: Hitters (ISLP) — 19 baseball predictors
### Time: ~50 minutes

## 🎯 Part 1 — Why Variable Selection?

With p predictors, using all of them can overfit — especially when p is large relative to n. Including irrelevant predictors:
- Increases variance of estimates
- Reduces interpretability
- May worsen prediction on new data

We want the smallest model that explains the data well.

In [ ]:
# Forward stepwise selection (greedy — each step adds best remaining feature)
def forward_stepwise(X, y, feature_names, max_features=None):
    if max_features is None: max_features = X.shape[1]
    selected = []
    remaining = list(range(X.shape[1]))
    results = []
    lr = LinearRegression()
    for step in range(max_features):
        best_score, best_feat = -np.inf, None
        for feat in remaining:
            candidate = selected + [feat]
            cv = cross_val_score(lr, X[:, candidate], y, cv=5,
                                scoring='neg_mean_squared_error').mean()
            if cv > best_score:
                best_score, best_feat = cv, feat
        selected.append(best_feat)
        remaining.remove(best_feat)
        results.append({'n_features': step+1, 'features': selected.copy(),
                       'cv_mse': -best_score,
                       'last_added': feature_names[best_feat]})
    return pd.DataFrame(results)

print('Running forward stepwise selection...')
fwd_results = forward_stepwise(X, y, features, max_features=15)
print(fwd_results[['n_features','last_added','cv_mse']].to_string())
print(f'\nBest model size: {fwd_results["cv_mse"].idxmin()+1} features')

In [ ]:
# Plot: CV MSE vs number of features
fig, ax = plt.subplots(figsize=(10, 4))
best_n = fwd_results['cv_mse'].idxmin()
ax.plot(fwd_results['n_features'], fwd_results['cv_mse'], 'o-', color='#1e5fa8', lw=2, markersize=6)
ax.axvline(fwd_results.loc[best_n, 'n_features'], color='#e85d20', lw=1.5, ls='--',
          label=f'Best: {fwd_results.loc[best_n,"n_features"]} features (CV MSE={fwd_results.loc[best_n,"cv_mse"]:.4f})')
ax.set_xlabel('Number of features')
ax.set_ylabel('5-fold CV MSE (log salary)')
ax.set_title('Forward Stepwise Selection — Hitters Dataset')
ax.legend()
plt.tight_layout(); plt.show()
best_features = fwd_results.loc[best_n, 'features']
print(f'\nSelected features: {[features[i] for i in best_features]}')

In [ ]:
# Information criteria: AIC, BIC, adjusted R²
# (computed on full training data — no cross-validation needed)
import statsmodels.api as sm

n = len(y)
X_sm = sm.add_constant(X[:, :10])  # first 10 features for illustration
model_sm = sm.OLS(y, X_sm).fit()

print('Model selection criteria (statsmodels):')  
print(f'  AIC:  {model_sm.aic:.2f}')
print(f'  BIC:  {model_sm.bic:.2f}')
print(f'  R²:   {model_sm.rsquared:.4f}')
print(f'  Adj R²: {model_sm.rsquared_adj:.4f}')
print()
print('Building models of increasing size and comparing criteria...')
for k in [1, 3, 5, 8, 10, 12, 15]:
    if k <= len(features):
        Xk = sm.add_constant(X[:, :k])
        mk = sm.OLS(y, Xk).fit()
        print(f'  p={k:<3} AIC={mk.aic:.1f}  BIC={mk.bic:.1f}  AdjR²={mk.rsquared_adj:.4f}')

In [ ]:
# Exercise workspace
# Task 1: Implement backward stepwise selection (start with all features, remove worst one each step)
# YOUR CODE HERE

# Task 2: Compare forward stepwise vs Lasso (from regularization notebook) on Hitters
# Which selects fewer features? Which has lower CV MSE?
from sklearn.linear_model import LassoCV
# YOUR CODE HERE

# Task 3: What features does the best 5-feature model select? Interpret each one
best_5 = fwd_results.loc[fwd_results['n_features']==5, 'features'].values[0]
print('Top 5 features:', [features[i] for i in best_5])
# YOUR CODE HERE — fit and report coefficients

In [ ]:
answers = {
    'q1': '',  # What is best subset selection and why is it computationally expensive?
    'q2': '',  # What is the difference between forward and backward stepwise selection?
    'q3': '',  # What does BIC penalize compared to AIC?
    'q4': '',  # Why is adjusted R² better than R² for model comparison?
    'q5': '',  # When would you use stepwise selection vs Lasso for variable selection?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')